# Maximum Sharpe ratio portfolio: the best return per unit of risk

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProvenDecisions/optimization-examples/blob/main/notebooks/06-finance/maximum-sharpe-ratio.ipynb)

Solved and checked by [ProvenDecisions](https://proven-decisions.ai/examples/maximum-sharpe-ratio?ref=github): the step-by-step walkthrough, the dual or KKT certificate and a version you can change and re-solve are on the example's page.

## The problem

Three assets have excess returns 8%, 12% and 6%, volatilities 15%, 25% and 10%, and correlations 0.3 (1-2), 0.1 (1-3), 0.2 (2-3). With no short sales, which portfolio has the highest Sharpe ratio?

## The model

Each line has a plain-language note.

$$
\begin{aligned}\max\;& \frac{0.08 w_{1} + 0.12 w_{2} + 0.06 w_{3}}{\sqrt{0.0225 w_{1}^{2} + 0.0225 w_{1} \cdot w_{2} + 0.003 w_{1} \cdot w_{3} + 0.0625 w_{2}^{2} + 0.01 w_{2} \cdot w_{3} + 0.01 w_{3}^{2}}} &&\text{make this as large as possible}\\ \text{s.t.}\;& w_{1} + w_{2} + w_{3} = 1 &&\text{limit 1: left side must equal 1}\\ \;& w_{1} \ge 0 &&\text{fraction of portfolio in asset 1 (excess return 8\%, volatility 15\%) cannot be negative}\\ \;& w_{2} \ge 0 &&\text{fraction of portfolio in asset 2 (excess return 12\%, volatility 25\%) cannot be negative}\\ \;& w_{3} \ge 0 &&\text{fraction of portfolio in asset 3 (excess return 6\%, volatility 10\%) cannot be negative}\end{aligned}
$$

## Solve it with cvxpy

Run the cell (in Colab, `cvxpy` is installed by the first line).

In [ ]:
%pip install -q cvxpy highspy

In [1]:
import cvxpy as cp
import numpy as np

# Charnes-Cooper: y = t*(w1, w2, w3) and t >= 0 turn the ratio into a convex problem
y = cp.Variable(3)  # y[i] = t * (w1, w2, w3)[i]
t = cp.Variable(nonneg=True)
constraints = [
    y[0] + y[1] + y[2] - t == 0,
    y[0] >= 0,
    y[1] >= 0,
    y[2] >= 0,
    cp.norm(np.array([[0.15, 0.075, 0.01], [0, 0.238484800354, 0.0178208422243], [0, 0, 0.0978898236918]]) @ y) <= 1,  # the square-root denominator, scaled to at most 1
    # (the matrix above is L, with L'L = the quadratic form under the square root)
]
objective = cp.Maximize(0.08*y[0] + 0.12*y[1] + 0.06*y[2])
problem = cp.Problem(objective, constraints)
problem.solve(solver='CLARABEL')
print(problem.status, problem.value)  # the best ratio
for name, v in zip(['w1', 'w2', 'w3'], y.value / t.value):
    print(name, '=', v)

optimal 0.8027209066782952
w1 = 0.3068116218378164
w2 = 0.11654891705446951
w3 = 0.576639461107714


## The checked answer

You asked for: the optimal portfolio weights w1, w2, w3 (the portfolio with the highest Sharpe ratio). The largest objective value is **0.802721**, at w1 = 0.306812, w2 = 0.116549, w3 = 0.576639. All 4 constraints are satisfied exactly. **Optimal, numerically checked.**

**Problem class:** [Ratio of linear to norm](https://proven-decisions.ai/problem-classes?ref=github); **solver:** CLARABEL (Charnes-Cooper SOCP). What makes the answer proven: a [dual or KKT certificate](https://proven-decisions.ai/glossary?ref=github), explained in plain words.

Step-by-step walkthrough, the proof of optimality and more: [Maximum Sharpe ratio portfolio: the best return per unit of risk](https://proven-decisions.ai/examples/maximum-sharpe-ratio?ref=github)

**Change the numbers and solve it free:** [this model in the ProvenDecisions solver](https://proven-decisions.ai/?try=maximize%20%280.08%2Aw1%20%2B%200.12%2Aw2%20%2B%200.06%2Aw3%29%20/%20sqrt%280.0225%2Aw1%5E2%20%2B%200.0625%2Aw2%5E2%20%2B%200.01%2Aw3%5E2%20%2B%200.0225%2Aw1%2Aw2%20%2B%200.003%2Aw1%2Aw3%20%2B%200.01%2Aw2%2Aw3%29%0Asubject%20to%0Aw1%20%2B%20w2%20%2B%20w3%20%3D%201%0Aw1%2C%20w2%2C%20w3%20%3E%3D%200%0A%23%20w1%20%3D%20fraction%20of%20portfolio%20in%20asset%201%20%28excess%20return%208%25%2C%20volatility%2015%25%29%0A%23%20w2%20%3D%20fraction%20of%20portfolio%20in%20asset%202%20%28excess%20return%2012%25%2C%20volatility%2025%25%29%0A%23%20w3%20%3D%20fraction%20of%20portfolio%20in%20asset%203%20%28excess%20return%206%25%2C%20volatility%2010%25%29%0A%23%20asked%3A%20the%20optimal%20portfolio%20weights%20w1%2C%20w2%2C%20w3%20%28the%20portfolio%20with%20the%20highest%20Sharpe%20ratio%29&ref=github)